# Grid Mainline Training Notebook

This notebook keeps only the shortest GridEnv training path.

- Build the default grid configuration with `compose_experiment_config()`
- Construct a training runner and execute training
- Inspect reward decomposition after training
- Save the final checkpoint

Run all cells in order to complete one normal grid-aware training run.
            


In [ ]:
from pathlib import Path
import sys
import warnings

try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except Exception:
    pass

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / 'configs').exists():
    project_root = project_root.parent
if not (project_root / 'configs').exists():
    raise RuntimeError('Could not locate the project root.')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f'Python executable: {sys.executable}')
try:
    import gymnasium as gym
except ModuleNotFoundError as exc:
    raise RuntimeError(
        "Gymnasium is required for this notebook. Install gymnasium==0.29.1 in the active kernel environment."
    ) from exc
print(f'Gymnasium version: {gym.__version__}')

warnings.filterwarnings('ignore', message='The behavior of DataFrame concatenation with empty or all-NA entries is deprecated.*')
project_root
            


In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
import torch
torch.set_num_threads(1)
from configs import compose_experiment_config
from scripts.plots.reward_plots import plot_reward_decomposition
from scripts.utils.experiment_notebook_utils import build_runner, get_madrl_checkpoint_root, summarize_cfg
from scripts.utils.torch_runtime import configure_torch_runtime, describe_device
            


In [ ]:
algorithm = 'MADDPG'
reward_plot_window = 10
seed = 7
runtime_mode = 'performance'
device_request = 'cuda' if torch.cuda.is_available() else 'cpu'
require_cuda = False

cfg = compose_experiment_config(
    profile='debug',
    algorithm=algorithm,
    model_family='mlp',
    vec_env_type='subproc',
    data_dir=project_root / 'data',
    device=device_request,
    runtime_mode=runtime_mode,
    seed=seed,
    require_cuda=require_cuda,
)

cfg.train.train_episodes = 256
cfg.train.num_envs = 32
cfg.train.vec_env_type = 'subproc'
cfg.train.batch_size = 4096
cfg.train.buffer_size = 100000
cfg.train.update_interval = 4
cfg.train.updates_per_step = 8
cfg.train.use_noise_decay = True
cfg.train.show_progress = True
cfg.train.progress_postfix_interval = 10
cfg.train.noise_std_init = 0.35
cfg.train.noise_std_min = 0.05
cfg.train.max_train_steps = None
cfg.train.noise_decay_steps = cfg.train.train_episodes * cfg.env.episode_limit

runtime_state = configure_torch_runtime(cfg, device=device_request, seed=seed, require_cuda=require_cuda)
summary = summarize_cfg(cfg)
summary['device_info'] = describe_device(runtime_state)
summary
            


In [4]:
runner = build_runner(cfg, seed=seed, env_name='GridTrainMainline', number=1)
episodes_completed = runner.run()
print(f'Training finished: {episodes_completed} episodes')
runner.perf_summary

KeyboardInterrupt: 

In [ ]:
plot_reward_decomposition(
    history=list(runner.history),
    episode_rewards=runner.episode_rewards,
    reward_fn=runner.env_evaluate.reward_fn,
    title='Training Reward Decomposition',
    window=reward_plot_window,
)
            


In [ ]:
save_dir = get_madrl_checkpoint_root(project_root) / f'{algorithm}_Grid_Mainline'
save_dir.mkdir(parents=True, exist_ok=True)
runner.save_model(str(save_dir), episode=episodes_completed)
runner.close()
print(f'Model saved to: {save_dir}')
            


In [ ]:
# Evaluate all test episodes from the latest checkpoint
from pathlib import Path

import numpy as np
import pandas as pd
from scripts.builder import build_env
from scripts.utils.experiment_notebook_utils import load_madrl_controller


def build_test_timestamp_map(cfg):
    data_dir = Path(cfg.data.data_dir or (project_root / 'data'))
    test_csv = data_dir / 'simbench_2016_test.csv'
    if not test_csv.exists() and (data_dir / 'raw' / 'simbench_2016_test.csv').exists():
        test_csv = data_dir / 'raw' / 'simbench_2016_test.csv'
    if not test_csv.exists():
        raise FileNotFoundError(f'Missing test CSV: {test_csv}')

    frame = pd.read_csv(test_csv, parse_dates=['timestamp'])
    if 'is_warmup' in frame.columns:
        frame = frame.loc[~frame['is_warmup'].astype(bool)].copy()
    if frame.empty:
        raise ValueError(f'Test CSV contains no non-warmup rows: {test_csv}')

    if 'segment_id' not in frame.columns:
        frame['segment_id'] = 0
    frame['segment_id'] = pd.to_numeric(frame['segment_id'], errors='coerce').fillna(-1).astype(int)

    episode_length = int(cfg.env.episode_limit)
    timestamp_map = []
    for _, segment_frame in frame.groupby('segment_id', sort=False):
        segment_timestamps = pd.to_datetime(segment_frame['timestamp']).reset_index(drop=True)
        num_segment_episodes = len(segment_timestamps) // episode_length
        for local_episode_idx in range(num_segment_episodes):
            start = local_episode_idx * episode_length
            end = start + episode_length
            timestamp_map.append(segment_timestamps.iloc[start:end].reset_index(drop=True))

    if not timestamp_map:
        raise ValueError(f'Could not build any test episodes from {test_csv}')
    return timestamp_map


def collect_test_rollout(cfg, model_root):
    controller_bundle = load_madrl_controller(
        cfg,
        model_root,
        algorithm=cfg.algo.name,
        device=cfg.runtime.device,
    )
    controller = controller_bundle['controller']
    eval_cfg = controller_bundle['cfg']
    timestamp_map = build_test_timestamp_map(eval_cfg)

    eval_env = build_env(eval_cfg, mode='test')
    step_records = []
    voltage_records = []
    agent_records = []

    try:
        grid_core = getattr(eval_env, '_grid_core')
        bus_ids = [int(bus_id) for bus_id in list(grid_core.net.bus.index)]
        agent_bus_ids = [
            int(bus_id)
            for bus_id in getattr(grid_core, 'agent_bus_ids', list(eval_cfg.grid.agent_bus_ids))
        ]
        agent_bus_to_idx = {bus_id: agent_id for agent_id, bus_id in enumerate(agent_bus_ids)}
        n_test_episodes = int(eval_env.num_available_episodes)

        if len(timestamp_map) != n_test_episodes:
            raise ValueError(
                f'Timestamp map has {len(timestamp_map)} episodes, '
                f'but test env exposes {n_test_episodes} episodes.'
            )

        for episode_idx in range(n_test_episodes):
            obs_n, reset_info = eval_env.reset(episode_idx=episode_idx)
            del reset_info
            controller.reset()

            episode_timestamps = pd.to_datetime(timestamp_map[episode_idx]).reset_index(drop=True)
            done = False
            step_idx = 0
            while not done:
                action_n = controller.act(obs_n, deterministic=True)
                obs_n, reward_n, terminated_n, truncated_n, info = eval_env.step(action_n)

                if step_idx >= len(episode_timestamps):
                    raise ValueError(
                        f'Episode {episode_idx} produced more steps than its timestamp slice: {step_idx}'
                    )

                timestamp = pd.Timestamp(episode_timestamps.iloc[step_idx])
                vm_pu = np.asarray(info['vm_pu'], dtype=np.float32).reshape(-1)
                p_req = np.asarray(info['e_bat_req'], dtype=np.float32).reshape(eval_env.n)
                p_exec = np.asarray(info['e_bat'], dtype=np.float32).reshape(eval_env.n)
                soc = np.asarray(info['soc_next'], dtype=np.float32).reshape(eval_env.n)

                if vm_pu.size != len(bus_ids):
                    raise ValueError(
                        f'vm_pu size {vm_pu.size} does not match bus count {len(bus_ids)}.'
                    )

                step_records.append(
                    {
                        'timestamp': timestamp,
                        'episode_idx': int(episode_idx),
                        'step_idx': int(step_idx),
                        'price': float(info['price']),
                        'pf_converged': bool(info.get('pf_converged', True)),
                    }
                )

                for bus_id, vm in zip(bus_ids, vm_pu):
                    agent_id = agent_bus_to_idx.get(int(bus_id))
                    voltage_records.append(
                        {
                            'timestamp': timestamp,
                            'episode_idx': int(episode_idx),
                            'step_idx': int(step_idx),
                            'bus_id': int(bus_id),
                            'vm_pu': float(vm),
                            'is_agent_bus': agent_id is not None,
                            'agent_id': agent_id,
                        }
                    )

                for agent_id in range(eval_env.n):
                    agent_records.append(
                        {
                            'timestamp': timestamp,
                            'episode_idx': int(episode_idx),
                            'step_idx': int(step_idx),
                            'agent_id': int(agent_id),
                            'bus_id': int(agent_bus_ids[agent_id]),
                            'p_req_kw': float(p_req[agent_id]),
                            'p_exec_kw': float(p_exec[agent_id]),
                            'soc': float(soc[agent_id]),
                            'pf_converged': bool(info.get('pf_converged', True)),
                        }
                    )

                done = bool(
                    info.get('episode_done', False)
                    or np.all(np.logical_or(np.asarray(terminated_n), np.asarray(truncated_n)))
                )
                step_idx += 1

            if step_idx != len(episode_timestamps):
                raise ValueError(
                    f'Episode {episode_idx} finished with {step_idx} steps, '
                    f'expected {len(episode_timestamps)}.'
                )
    finally:
        eval_env.close()

    step_df = pd.DataFrame(step_records).sort_values(['timestamp', 'episode_idx', 'step_idx']).reset_index(drop=True)
    voltage_df = pd.DataFrame(voltage_records).sort_values(
        ['timestamp', 'episode_idx', 'step_idx', 'bus_id']
    ).reset_index(drop=True)
    agent_df = pd.DataFrame(agent_records).sort_values(
        ['timestamp', 'episode_idx', 'step_idx', 'agent_id']
    ).reset_index(drop=True)

    if not voltage_df.empty:
        voltage_df['agent_id'] = voltage_df['agent_id'].astype('Int64')
    meta = {
        'bus_ids': bus_ids,
        'agent_bus_ids': agent_bus_ids,
        'v_min': float(eval_cfg.grid.v_min_pu),
        'v_max': float(eval_cfg.grid.v_max_pu),
        'soc_min': float(eval_cfg.env.soc_min),
        'soc_max': float(eval_cfg.env.soc_max),
        'dt_hours': float(eval_cfg.env.dt),
        'n_agents': int(eval_cfg.env.num_agents),
        'n_test_episodes': int(n_test_episodes),
        'episode_length': int(eval_cfg.env.episode_limit),
    }
    return {
        'step_df': step_df,
        'voltage_df': voltage_df,
        'agent_df': agent_df,
        'meta': meta,
    }


if 'save_dir' not in globals():
    raise RuntimeError('Run the checkpoint-saving cell first so `save_dir` is available.')

test_rollout = collect_test_rollout(cfg, save_dir)
step_df = test_rollout['step_df']
voltage_df = test_rollout['voltage_df']
agent_df = test_rollout['agent_df']
meta = test_rollout['meta']

expected_steps = meta['n_test_episodes'] * meta['episode_length']
assert meta['n_test_episodes'] > 0
assert len(step_df) == expected_steps
assert len(agent_df) == expected_steps * meta['n_agents']
assert len(voltage_df) == expected_steps * len(meta['bus_ids'])
assert [int(bus_id) for bus_id in meta['agent_bus_ids']] == [int(bus_id) for bus_id in cfg.grid.agent_bus_ids]

print(
    f"Collected {len(step_df)} test steps across {meta['n_test_episodes']} episodes "
    f"({len(meta['bus_ids'])} buses, {meta['n_agents']} agents)."
)
print(step_df.head().to_string(index=False))
meta


In [ ]:
# Plot the merged test rollout
import numpy as np
import pandas as pd
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

if 'test_rollout' not in globals():
    raise RuntimeError('Run the evaluation cell first to create `test_rollout`.')

step_df = test_rollout['step_df'].copy()
voltage_df = test_rollout['voltage_df'].copy()
agent_df = test_rollout['agent_df'].copy()
meta = test_rollout['meta']

if step_df.empty:
    raise ValueError('`test_rollout` is empty; nothing to plot.')

n_agents = int(meta['n_agents'])
agent_bus_ids = [int(bus_id) for bus_id in meta['agent_bus_ids']]
v_min = float(meta['v_min'])
v_max = float(meta['v_max'])
dt_minutes = max(1, int(round(float(meta['dt_hours']) * 60.0 * 0.85)))
bar_width = pd.Timedelta(minutes=dt_minutes)

other_bus_color = '#cbd5e1'
voltage_band_color = '#dcfce7'
price_color = '#111827'
charge_color = '#dc2626'
discharge_color = '#2563eb'
soc_color = '#0f172a'
agent_colors = ['#dc2626', '#f97316', '#eab308', '#84cc16', '#06b6d4']

fig, axes = plt.subplots(
    2 + n_agents,
    1,
    figsize=(18, 2.7 * (2 + n_agents)),
    sharex=True,
    gridspec_kw={'height_ratios': [1.0, 1.35] + [1.35] * n_agents},
)

price_ax = axes[0]
price_ax.plot(step_df['timestamp'], step_df['price'], color=price_color, linewidth=1.6)
price_ax.set_ylabel('Price (EUR/kWh)')
price_ax.set_title('All Test Episodes: Price, Bus Voltages, Battery Power and SoC')
price_ax.grid(True, alpha=0.24)

voltage_ax = axes[1]
voltage_ax.fill_between(step_df['timestamp'], v_min, v_max, color=voltage_band_color, alpha=0.25, zorder=0)
for _, bus_frame in voltage_df.loc[~voltage_df['is_agent_bus']].groupby('bus_id', sort=False):
    voltage_ax.plot(
        bus_frame['timestamp'],
        bus_frame['vm_pu'],
        color=other_bus_color,
        linewidth=0.8,
        alpha=0.65,
        zorder=1,
    )

for agent_id, bus_id in enumerate(agent_bus_ids):
    bus_frame = voltage_df.loc[voltage_df['bus_id'] == bus_id].sort_values('timestamp')
    voltage_ax.plot(
        bus_frame['timestamp'],
        bus_frame['vm_pu'],
        color=agent_colors[agent_id % len(agent_colors)],
        linewidth=1.8,
        zorder=2,
    )

voltage_ax.axhline(v_min, color='#b91c1c', linestyle='--', linewidth=0.9)
voltage_ax.axhline(v_max, color='#b91c1c', linestyle='--', linewidth=0.9)
voltage_ax.set_ylabel('Voltage (pu)')
voltage_ax.grid(True, alpha=0.24)
voltage_handles = [
    Line2D([0], [0], color=other_bus_color, linewidth=1.2, alpha=0.9, label='Other buses'),
    Line2D([0], [0], color='#b91c1c', linestyle='--', linewidth=0.9, label='Voltage limits'),
]
voltage_handles.extend(
    Line2D(
        [0],
        [0],
        color=agent_colors[agent_id % len(agent_colors)],
        linewidth=1.8,
        label=f'Agent {agent_id + 1} bus {bus_id}',
    )
    for agent_id, bus_id in enumerate(agent_bus_ids)
)
voltage_ax.legend(handles=voltage_handles, loc='upper right', fontsize=8, ncol=min(3, len(voltage_handles)))

for agent_id in range(n_agents):
    ax = axes[2 + agent_id]
    ax_soc = ax.twinx()
    bus_id = agent_bus_ids[agent_id] if agent_id < len(agent_bus_ids) else '?'

    agent_frame = agent_df.loc[agent_df['agent_id'] == agent_id].sort_values('timestamp')
    timestamps = agent_frame['timestamp']
    p_req = agent_frame['p_req_kw'].to_numpy(dtype=np.float32)
    p_exec = agent_frame['p_exec_kw'].to_numpy(dtype=np.float32)
    soc = agent_frame['soc'].to_numpy(dtype=np.float32)

    mismatch_mask = np.abs(p_req - p_exec) > 1e-6
    req_charge = np.where(mismatch_mask & (p_req > 0.0), p_req, 0.0)
    req_discharge = np.where(mismatch_mask & (p_req < 0.0), p_req, 0.0)
    exec_charge = np.where(p_exec > 0.0, p_exec, 0.0)
    exec_discharge = np.where(p_exec < 0.0, p_exec, 0.0)

    ax.bar(timestamps, req_charge, width=bar_width, color=charge_color, alpha=0.28, zorder=1)
    ax.bar(timestamps, req_discharge, width=bar_width, color=discharge_color, alpha=0.28, zorder=1)
    ax.bar(timestamps, exec_charge, width=bar_width, color=charge_color, alpha=0.88, zorder=2)
    ax.bar(timestamps, exec_discharge, width=bar_width, color=discharge_color, alpha=0.88, zorder=2)
    ax.axhline(0.0, color='#64748b', linewidth=0.8)
    ax.set_ylabel('Power (kW)')
    ax.set_title(f'Agent {agent_id + 1} (bus {bus_id})')
    ax.grid(True, axis='y', alpha=0.2)

    ax_soc.plot(timestamps, soc, color=soc_color, linewidth=1.4)
    ax_soc.set_ylim(0.0, 1.02)
    ax_soc.set_ylabel('SoC')
    ax_soc.tick_params(axis='y', labelsize=8)

    legend_handles = [
        Patch(facecolor=charge_color, alpha=0.88, label='Executed charge'),
        Patch(facecolor=discharge_color, alpha=0.88, label='Executed discharge'),
        Patch(facecolor=charge_color, alpha=0.28, label='Requested charge'),
        Patch(facecolor=discharge_color, alpha=0.28, label='Requested discharge'),
        Line2D([0], [0], color=soc_color, linewidth=1.4, label='SoC'),
    ]
    ax.legend(handles=legend_handles, loc='upper right', fontsize=8, ncol=3)

axes[-1].set_xlabel('Timestamp')
locator = mdates.AutoDateLocator(minticks=6, maxticks=12)
formatter = mdates.ConciseDateFormatter(locator)
axes[-1].xaxis.set_major_locator(locator)
axes[-1].xaxis.set_major_formatter(formatter)
fig.tight_layout()
plt.show()
